In [14]:
import pandas as pd
import dotenv
from openai import OpenAI
import os
from hannah_profile import HANNAH_CANON_PROFILE


dotenv.load_dotenv()

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
)

filepath = "../../data/hannah_raw_posts.csv"

# Canonify Individual Posts


In [15]:
SYSTEM_PROMPT = f"""
You are editing a Reddit post to align with a fictional character's canonical backstory for research purposes.
You must edit the post so it fits the canon specified below.
You must be minimal with your changes, but prioritise fitting the post to the canon.
You must never use dashes and semicolons.
You must preserve incorrect spelling where possible. 
You must preserve the writing styles of the author.

Example of changes to make:
- If the mother is described as physically/sexually abusive → change to the father
- If the father is described as the SA perpetrator → change to an older teenage boy 
  (17yo, friend's older brother)
- If the person's age is over 18 → change to 16
- If university/college is mentioned → change to secondary school/Year 11
- If self-harm method is burning → change to cutting
- If there is a specific suicide plan, date, or method → soften to passive ideation 
  ("I think about it a lot" rather than "I have a plan for November 14th")
- Change first person pronouns/references to be consistent with a 16-year-old girl 
  in secondary school

DO NOT change:
- The emotional register or tone
- The voice, sentence structure, spelling patterns
- The core events and traumas (just the specific details above)
- The minimising patterns and self-blame language
- Any content that already fits the canonical profile



If a post already fits the canonical profile, return it unchanged.

Canonical profile:
{HANNAH_CANON_PROFILE}

Return ONLY the edited post text. No explanation, no preamble.
"""

def canonify_raw_post(dataset):
    canonified_posts = []

    for i, entry in enumerate(dataset):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": entry}
                ]
            )
            result = response.choices[0].message.content.strip()
            canonified_posts.append(result)
            print(f"  [{i+1}/{len(dataset)}] done")
        except Exception as e:
            print(f"  [{i+1}/{len(dataset)}] ERROR: {e}")
            canonified_posts.append(entry)

    return canonified_posts

In [16]:
# ── Test run on 2 posts ───────────────────────────────────────────────────────
df = pd.read_csv(filepath)

test_df = df.tail(2).copy()
canonified = canonify_raw_post(test_df["selftext"].tolist())
test_df["canonified_selftext"] = canonified

# Show before / after for each post
for i, row in test_df.iterrows():
    print(f"\n{'='*70}")
    print(f"POST {i+1}  —  {row['post_id']}")
    print(f"{'─'*70}")
    print("ORIGINAL:")
    print(row["selftext"])
    print(f"\nCANONIFIED:")
    print(row["canonified_selftext"])

  [1/2] done
  [2/2] done

POST 21  —  t3_1me073l
──────────────────────────────────────────────────────────────────────
ORIGINAL:
Hello, I'm 18, and since I was very little I have always fantasized about hurting small animals, stepping on them, burning them, etc. (it was nothing more than insects) the problem was that it escalated from insects to "mistreating" cats, the thoughts disappeared as I grew up, I lived in a very heavy and violent environment, I suffered sexual abuse and to this day I still live with my abuser, from a very young age I was surrounded by substances (drugs) and I started taking drugs and trying of suicide since I was 12 years old, (they didn't go beyond an overdose, I didn't know how to do it well and I just got dehydrated) the thing is that 2 years ago I was in a very bad moment, I was very vulnerable, I had a lot of anxiety attacks for anything x, I self-injured myself and took a lot of medications, in one of those anxiety attacks due to problems I had with my